In [48]:
import pandas as pd
import sqlite3

In [49]:
# conectamos con la base de datos my_database.db
connection = sqlite3.connect("database_cervezas.db")

# obtenemos un cursor que utilizamos para las queries
crsr = connection.cursor()

In [50]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [51]:
res = crsr.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

cervezas
bares
empleados
reparto


In [52]:
# crea las tablas


In [53]:
query = '''
CREATE TABLE IF NOT EXISTS cervezas (
    CodC VARCHAR(2),
    Envase VARCHAR(32),
    Capacidad FLOAT(2),
    Stock INT(5),
    PRIMARY KEY (CodC)
)
'''

crsr.execute(query)

In [54]:
query = """
INSERT INTO cervezas VALUES
('01','Botella',0.2,3600),
('02','Botella',0.33,1200),
('03','Lata',0.33,2400),
('04','Botella',1,288),
('05','Barril',60,30)
"""
crsr.execute(query)

OperationalError: database is locked

In [ ]:
query = '''
CREATE TABLE IF NOT EXISTS bares (
    CodB VARCHAR(2),
    Nombre VARCHAR(32),
    Cif VARCHAR(32),
    Localidad VARCHAR(32),
    PRIMARY KEY (CodB)
)
'''

crsr.execute(query)

In [ ]:
query = """
INSERT INTO bares VALUES
('001','Stop','11111111X','Villa Botijo'),
('002','Las Vegas','22222222Y','Villa Botijo'),
('003','Club Social',NULL,'Las Ranas'),
('004','Otra Ronda','33333333Z','La Esponja')
"""
crsr.execute(query)

In [ ]:
query = '''
CREATE TABLE IF NOT EXISTS empleados (
    CodE VARCHAR(2),
    Nombre VARCHAR(32),
    Sueldo INT(5),
    PRIMARY KEY (CodE)
)
'''

crsr.execute(query)

In [ ]:
query = """
INSERT INTO empleados VALUES
(1,'Carlos Lopez',120000),
(2,'Rosa Perez',100000),
(3,'Luisa Garcia',100000)
"""
crsr.execute(query)

In [ ]:
query = '''
CREATE TABLE IF NOT EXISTS reparto (
	CodE VARCHAR(2) NOT NULL,
	CodB VARCHAR(3) NOT NULL,
	CodC VARCHAR(2) NOT NULL,
	Fecha DATE NOT NULL,
	Cantidad SMALLINT,
	PRIMARY KEY (CodE,CodB,CodC,fecha),
	FOREIGN KEY (CodE) REFERENCES empleados(CodE),
    FOREIGN KEY (CodB) REFERENCES bares(CodB),
    FOREIGN KEY (CodC) REFERENCES cervezas(CodC)

);
'''
crsr.execute(query)

In [ ]:
query = """
INSERT INTO reparto VALUES
(1,'001','01','10/21/05',240),
(1,'001','02','10/21/05',48),
(1,'002','03','10/22/05',60),
(1,'004','05','10/22/05',4),
(2,'002','03','10/22/05',48),
(2,'002','05','10/23/05',2),
(2,'004','01','10/23/05',480),
(2,'004','02','10/24/05',72),
(3,'003','03','10/24/05',48),
(3,'003','04','10/25/05',20)
"""
crsr.execute(query)

In [ ]:
connection.commit()

In [ ]:
# 1 Obtener el nombre de los empleados que hayan repartido al bar Stop durante la semana 
# del 17 al 23 de octubre de 2005.

query = """
SELECT DISTINCT e.Nombre
FROM reparto r
JOIN empleados e ON r.CodE = e.CodE
JOIN bares b ON r.CodB = b.CodB
WHERE b.Nombre = 'Stop'
AND r.Fecha BETWEEN '10/17/05' AND '10/23/05' 
"""
sql_query(query)

,Nombre
0,Carlos Lopez


In [ ]:
# 2 Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo Botella y
# capacidad inferior a 1 litro, ordenados por localidad

query = '''
SELECT DISTINCT bares.Cif, bares.Nombre
FROM reparto
LEFT JOIN bares
ON reparto.CodB = bares.CodB
LEFT JOIN cervezas
ON reparto.CodC = cervezas.CodC
WHERE cervezas.Envase = "Botella" AND cervezas.Capacidad < 1
ORDER BY bares.Localidad
'''

sql_query(query)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


In [ ]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad)

query = '''
SELECT DISTINCT bares.Nombre, cervezas.Envase, cervezas.Capacidad, reparto.Fecha, reparto.Cantidad
FROM reparto
LEFT JOIN bares
ON reparto.CodB = bares.CodB
LEFT JOIN cervezas
ON reparto.CodC = cervezas.CodC
LEFT JOIN empleados
ON reparto.CodE = empleados.CodE
WHERE empleados.nombre = "Carlos Lopez"
'''

sql_query(query)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,10/21/05,240
1,Stop,Botella,0.33,10/21/05,48
2,Las Vegas,Lata,0.33,10/22/05,60
3,Otra Ronda,Barril,60.00,10/22/05,4


In [ ]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo botella y capacidad 0.2 o
# 0.33

query = '''
SELECT DISTINCT bares.Nombre, cervezas.Envase, cervezas.Capacidad
FROM reparto
LEFT JOIN bares
ON reparto.CodB = bares.CodB
LEFT JOIN cervezas
ON reparto.CodC = cervezas.CodC
WHERE cervezas.Envase = "Botella" AND cervezas.Capacidad IN (0.2, 0.33)
'''
sql_query(query)

,Nombre,Envase,Capacidad
0,Stop,Botella,0.20
1,Stop,Botella,0.33
2,Otra Ronda,Botella,0.20
3,Otra Ronda,Botella,0.33


In [ ]:
# 5. Nombre de los empleados que han repartido a los bares "Stop" y "Las Vegas" cervezas con
# envase botella.

query = '''
SELECT DISTINCT bares.Nombre "Nombre bar", empleados.Nombre "Nombre empleado", cervezas.Envase
FROM reparto
LEFT JOIN bares
ON reparto.CodB = bares.CodB
LEFT JOIN cervezas
ON reparto.CodC = cervezas.CodC
LEFT JOIN empleados
ON reparto.CodE = empleados.CodE
WHERE bares.Nombre IN ("Stop", "Las Vegas") AND cervezas.Envase = "Botella"
'''
sql_query(query)

,Nombre bar,Nombre empleado,Envase
0,Stop,Carlos Lopez,Botella


In [ ]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de Villa
# Botijo.

query = '''
SELECT empleados.CodE, empleados.Nombre, COUNT(*) 'Viajes'
FROM reparto
LEFT JOIN bares
ON reparto.CodB = bares.CodB
LEFT JOIN empleados
ON reparto.CodE = empleados.CodE
WHERE bares.Localidad <> "Villa Botijo"
GROUP BY  empleados.CodE
'''
sql_query(query)

,CodE,Nombre,Viajes
0,1,Carlos Lopez,1
1,2,Rosa Perez,2
2,3,Luisa Garcia,2


In [ ]:
query = '''
SELECT bares.Nombre, bares.Localidad, SUM(cervezas.Capacidad * reparto.Cantidad) AS litros
FROM reparto
LEFT JOIN bares
ON reparto.CodB = bares.CodB
LEFT JOIN cervezas
ON reparto.CodC = cervezas.CodC
GROUP BY 1,2
ORDER BY 3 DESC
LIMIT 1
'''
sql_query(query)

,Nombre,Localidad,litros
0,Otra Ronda,La Esponja,359.76


In [ ]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

query = '''
SELECT b.Nombre, b.Localidad,
       SUM(r.Cantidad * c.Capacidad) as Litros
FROM reparto r
JOIN cervezas c ON r.CodC = c.CodC
JOIN bares b ON r.CodB = b.CodB
GROUP BY b.Nombre, b.Localidad
ORDER BY Litros DESC
LIMIT 1
'''
sql_query(query)

,Nombre,Localidad,litros
0,Otra Ronda,La Esponja,359.76


In [ ]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y
# capacidad menor que 1 litro.

query = '''
SELECT DISTINCT bares.Nombre
FROM reparto
LEFT JOIN bares
ON reparto.CodB = bares.CodB
LEFT JOIN cervezas
ON reparto.CodC = cervezas.CodC
WHERE cervezas.Envase = "Botella" AND cervezas.Capacidad < 1
'''
sql_query(query)

,Nombre
0,Stop
1,Otra Ronda


In [ ]:
#9. Subir un 5% el sueldo del empleado que más días haya trabajado.

query = '''
UPDATE empleados
SET Sueldo = Sueldo * 1.05
WHERE CodE = (
    SELECT CodE
    FROM reparto
    GROUP BY CodE
    ORDER BY COUNT(DISTINCT Fecha) DESC
    LIMIT 1
)
'''
sql_query(query)
connection.commit()

,Dias_trabajados,CodE,Nombre
0,3,2,Rosa Perez
1,2,3,Luisa Garcia
2,2,1,Carlos Lopez


In [ ]:
sql_query("SELECT * FROM empleados")

,t.Sueldo * 1.05
0,115500.0
